In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from patsy import dmatrix
from statsmodels.genmod.families import Gaussian
from statsmodels.genmod.generalized_linear_model import GLM
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
from statsmodels.genmod.families.links import Identity
from sklearn.feature_selection import SelectKBest, f_regression

In [ ]:
with open("data/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

true_y_list = []
glm_pred_y_list = []
r2_scores_glm = []

results_df = pd.DataFrame(columns=['Year', 'True_RET', 'Predicted_RET', 'R2_GLM'])

for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='year'):
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_train = train_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    glm_model = GLM(y_train, X_train_scaled, family=Gaussian())
    glm_results = glm_model.fit()
    
    glm_pred_y = glm_results.predict(X_test_scaled)
    
    r2_glm = r2_score(y_test, glm_pred_y)
    r2_scores_glm.append(r2_glm)
    
    true_y_list.append(y_test)
    glm_pred_y_list.append(glm_pred_y)
    
    year_results = pd.DataFrame({
        'Year': [year] * len(y_test),
        'True_RET': y_test,
        'Predicted_RET': glm_pred_y,
        'R2_GLM': [r2_glm] * len(y_test)
    })
    results_df = pd.concat([results_df, year_results], ignore_index=True)
    
    print(f'Year: {year}, R² (GLM): {r2_glm:.4f}')

average_r2_glm = np.mean(r2_scores_glm)
print(f'Average R² (GLM) over all years: {average_r2_glm:.4f}')

results_df.to_csv('data/glm_results.csv', index=False)

In [ ]:
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016
predictions = []
y_test_list = []
for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='year'):
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_train = train_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    selector = SelectKBest(score_func=f_regression, k=50)
    X_train_selected = selector.fit_transform(X_train_scaled, y_train)
    X_test_selected = selector.transform(X_test_scaled)
    selected_features = selector.get_support(indices=True)
    print(f'Selected features: {selected_features}')
    glm_model = GLM(y_train, X_train_selected, family=Gaussian(link=Identity())) 
    glm_results = glm_model.fit()
    glm_pred_y = glm_results.predict(X_test_selected)
    predictions.extend(glm_pred_y)
    y_test_list.extend(y_test)
    r2_glm = r2_score(y_test, glm_pred_y)
    print(f'Year: {year}, R² (GLM): {r2_glm:.4f}')
all_predictions = np.array(predictions)
all_y_test = np.array(y_test_list)

ss_res = np.sum((all_y_test - all_predictions) ** 2) 
ss_tot = np.sum((all_y_test) ** 2)  
total_r2_out = 1 - (ss_res / ss_tot)

print(f'Total Out of Sample R^2: {total_r2_out:.4f}')